# Consolidated Demo: Geopolitical Brief Pipeline

This notebook is a **step-by-step team walkthrough** showing exactly how data is sourced and transformed into:
- `geopolitical_brief.txt`
- `geopolitical_brief.json`

Run cells top to bottom.


## 0) Setup and imports
Resolves repo root, adds `src` to path, and imports only existing project helpers.


In [3]:
from __future__ import annotations

import json
import re
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait

_here = Path.cwd().resolve()
if (_here / "pyproject.toml").is_file():
    NEXUS_ROOT = _here
elif (_here.parent / "pyproject.toml").is_file():
    NEXUS_ROOT = _here.parent
else:
    raise RuntimeError(f"Cannot find pyproject.toml from cwd={_here}")

sys.path.insert(0, str(NEXUS_ROOT / "src"))

# --- SETUP: install runtime deps into THIS notebook kernel (so imports work) ---
# Avoid `pip install -e .` here because `pyproject.toml` requires Python>=3.11,
# while some local notebook kernels may run on 3.9.
import subprocess

_runtime_req = NEXUS_ROOT / "requirements.txt"
if _runtime_req.is_file():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(_runtime_req)],
        cwd=NEXUS_ROOT,
    )

_nb_req = NEXUS_ROOT / "requirements-notebooks.txt"
if _nb_req.is_file():
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(_nb_req)],
        cwd=NEXUS_ROOT,
    )

from nexus.ingest.rss import fetch_rss_as_news_items
from nexus.ingest.selenium_scraper import _build_driver, _clean_heading  # noqa: SLF001

print("NEXUS_ROOT:", NEXUS_ROOT)



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ERROR: Package 'nexus' requires a different Python: 3.9.12 not in '>=3.11'


CalledProcessError: Command '['/Users/alex1602e19/opt/anaconda3/bin/python', '-m', 'pip', 'install', '-q', '-e', '.']' returned non-zero exit status 1.

## 1) Demo configuration
Choose track and ticker fallback (Yahoo RSS).


In [ ]:
TRACK_ID = "agriculture"
TICKER = "MOS"  # e.g., MOS, CTVA, BG
MIN_HITS = 6
HEADLESS = False  # set True if you do not want a visible browser window

SELENIUM_PAGES = [
    ("bbc_business", "https://www.bbc.co.uk/news/business", "h3"),
    ("guardian_farming", "https://www.theguardian.com/environment/farming", "h3"),
]
RSS_FALLBACKS = [
    ("rss_bbc_business", "https://feeds.bbci.co.uk/news/business/rss.xml"),
    ("rss_bbc_world", "https://feeds.bbci.co.uk/news/world/rss.xml"),
]
YAHOO_RSS = "https://feeds.finance.yahoo.com/rss/2.0/headline?s={sym}&region=US&lang=en-US"


## 2) Load track and build keyword dictionary
This is the exact term set used to decide if a headline is relevant.


In [ ]:
tracks_doc = json.loads((NEXUS_ROOT / "config" / "investment_tracks.json").read_text(encoding="utf-8"))
track = next(t for t in tracks_doc["tracks"] if t["track_id"] == TRACK_ID)

kws = [str(k).strip().lower() for k in (track.get("keywords") or []) if str(k).strip()]
extra = ["monsanto", "bayer", "roundup", "glyphosate", "gmo", "seed", "seeds", "farm", "farming", "food", "crop", "commodities", "commodity", TICKER.lower()]
terms = frozenset(kws + extra)

print("Track:", track["label"], "[", track["track_id"], "]")
print("Companies:", ", ".join(f"{c['ticker']}:{c['name']}" for c in track['companies']))
print("Term count:", len(terms))
print("Sample terms:", sorted(list(terms))[:20])


## 3) Helper functions (matching, dedupe, source attribution)
All summaries are made only from scraped headlines.


In [ ]:
@dataclass
class BriefRow:
    headline: str
    url: str | None
    source_id: str

def norm_key(headline: str) -> str:
    return re.sub(r"\s+", " ", headline.strip().lower()[:220])

def row_matches(headline: str, terms: frozenset[str]) -> bool:
    h = headline.lower()
    return any(t and t in h for t in terms)

def snapshot_h3(driver, url: str, wait_css: str):
    driver.get(url)
    WebDriverWait(driver, 22).until(EC.presence_of_element_located((By.CSS_SELECTOR, wait_css)))
    time.sleep(0.75)
    return driver.execute_script("""
    return Array.from(document.querySelectorAll('h3')).map(function (h) {
      var t = (h.innerText || '').trim();
      var link = h.querySelector('a');
      if (!link) { link = h.closest('a'); }
      var href = (link && link.href) ? link.href : null;
      return { text: t, href: href };
    }).filter(function (x) { return x.text.length >= 12; });
    """)


## 4) Collect from Selenium sources first
This opens Chrome and extracts live DOM headlines from free sites.


In [ ]:
collected: list[BriefRow] = []
seen: set[str] = set()
counts: dict[str, int] = {}

driver = _build_driver(headless=HEADLESS, page_load_timeout=40)
try:
    for sid, url, wait_css in SELENIUM_PAGES:
        if len(collected) >= MIN_HITS:
            counts[sid] = 0
            continue
        before = len(collected)
        rows = snapshot_h3(driver, url, wait_css)
        for row in rows or []:
            head = _clean_heading((row.get("text") or "").strip())
            if len(head) < 16:
                continue
            nk = norm_key(head)
            if nk in seen or not row_matches(head, terms):
                continue
            seen.add(nk)
            href = row.get("href")
            if isinstance(href, str) and not href.strip():
                href = None
            collected.append(BriefRow(headline=head, url=href, source_id=sid))
        counts[sid] = len(collected) - before
        print(f"{sid}: +{counts[sid]} (total {len(collected)})")
finally:
    driver.quit()


## 5) Fallback: RSS sources, then ticker-specific Yahoo RSS
If Selenium is sparse, continue with RSS to avoid empty output.


In [ ]:
for sid, feed in RSS_FALLBACKS:
    if len(collected) >= MIN_HITS:
        counts.setdefault(sid, 0)
        continue
    before = len(collected)
    try:
        items = fetch_rss_as_news_items(feed, source_tag=sid, timeout=25.0)
    except Exception:
        counts[sid] = 0
        continue
    for it in items:
        head = _clean_heading((it.get("headline") or "").strip())
        if len(head) < 12:
            continue
        nk = norm_key(head)
        if nk in seen or not row_matches(head, terms):
            continue
        seen.add(nk)
        u = it.get("url")
        collected.append(BriefRow(headline=head, url=u if isinstance(u, str) else None, source_id=sid))
    counts[sid] = len(collected) - before
    print(f"{sid}: +{counts[sid]} (total {len(collected)})")

yid = f"rss_yahoo_{TICKER.upper()}"
if len(collected) < MIN_HITS:
    before = len(collected)
    try:
        items = fetch_rss_as_news_items(YAHOO_RSS.format(sym=TICKER.upper()), source_tag=yid, timeout=25.0)
    except Exception:
        items = []
    for it in items:
        head = _clean_heading((it.get("headline") or "").strip())
        if len(head) < 8:
            continue
        nk = norm_key(head)
        if nk in seen:
            continue
        seen.add(nk)
        u = it.get("url")
        collected.append(BriefRow(headline=head, url=u if isinstance(u, str) else None, source_id=yid))
    counts[yid] = len(collected) - before
    print(f"{yid}: +{counts[yid]} (total {len(collected)})")


## 6) Inspect collected evidence
Every row below is directly from source pages/feed items.


In [ ]:
print("Total collected:", len(collected))
print("Counts by source:", counts)
for i, r in enumerate(collected[:20], 1):
    print(f"{i:02d}. [{r.source_id}] {r.headline}")
    if r.url:
        print("    ", r.url)


## 7) Build JSON graph object from scraped rows
Only pairs with at least one matching scraped headline are emitted.


In [ ]:
def company_mentioned(headline: str, ticker: str, name: str) -> bool:
    t = (ticker or "").strip().upper()
    if not t:
        return False
    if len(t) <= 2 and re.search(rf"\b{re.escape(t)}\b", headline):
        return True
    if len(t) > 2 and re.search(rf"(?<![A-Za-z0-9]){re.escape(t)}(?![A-Za-z0-9])", headline, re.IGNORECASE):
        return True
    for w in re.findall(r"[A-Za-z]{5,}", name or ""):
        wl = w.lower()
        if wl in {"company","global","corporation","holdings","group","limited","inc"}:
            continue
        if re.search(rf"(?<![A-Za-z0-9]){re.escape(w)}(?![A-Za-z0-9])", headline, re.IGNORECASE):
            return True
    return False

nodes = [{"id": c["ticker"], "name": c["name"]} for c in track["companies"]]
links = []
for i in range(len(track["companies"])):
    for j in range(i + 1, len(track["companies"])):
        a, b = track["companies"][i], track["companies"][j]
        ta, tb = a["ticker"], b["ticker"]
        ra = [r for r in collected if company_mentioned(r.headline, ta, a["name"])]
        rb = [r for r in collected if company_mentioned(r.headline, tb, b["name"])]
        both = [r for r in collected if (r in ra and r in rb)]
        use_rows = both if both else (ra + [r for r in rb if r not in ra])
        if not use_rows:
            continue
        summary = " ".join(r.headline for r in use_rows[:6]).strip()
        seen_src = set()
        src = []
        for r in use_rows[:10]:
            s = r.url if (r.url and r.url.startswith("http")) else f"{r.source_id} | {r.headline}"
            if s not in seen_src:
                seen_src.add(s)
                src.append(s)
        links.append({
            "pair": f"{ta} - {tb}",
            "relationship": "both" if both else "one",
            "summary": summary,
            "sources": src,
        })

brief_json = {
    "track_name": track["label"],
    "last_updated": datetime.now(timezone.utc).date().isoformat(),
    "nodes": nodes,
    "links": links,
}
print("nodes:", len(nodes), "links:", len(links))
print(json.dumps(brief_json, indent=2, ensure_ascii=False)[:1400], "\n...")


## 8) Write final artifacts: TXT + JSON
This is the final formatting + file output stage.


In [ ]:
OUT_TXT = NEXUS_ROOT / "geopolitical_brief.txt"
OUT_JSON = NEXUS_ROOT / "geopolitical_brief.json"

ordered_ids = [s[0] for s in SELENIUM_PAGES] + [s[0] for s in RSS_FALLBACKS] + [f"rss_yahoo_{TICKER.upper()}"]
prov = [f"{sid}: +{counts.get(sid, 0)}" for sid in ordered_ids]

lines = [
    "NEXUS — Demo geopolitical/ag brief (consolidated walkthrough)",
    f"Primary ticker (Yahoo RSS): {TICKER.upper()}",
    f"Track: {track['label']} [{track['track_id']}]",
    f"Collected (UTC): {datetime.now(timezone.utc).replace(microsecond=0).isoformat().replace('+00:00','Z')}",
    f"Match threshold: min_hits={MIN_HITS} | terms≈{len(terms)}",
    "",
    "--- Provenance (matches per source) ---",
    *[f"• {x}" for x in prov],
    "",
    "--- Matched headlines ---",
]
if not collected:
    lines.append("(none after full chain)")
else:
    for r in collected:
        lines.append(f"• [{r.source_id}] {r.headline}")
        if r.url:
            lines.append(f"  {r.url}")

json_text = json.dumps(brief_json, indent=2, ensure_ascii=False)
lines.extend(["", "--- JSON (same object as geopolitical_brief.json) ---", json_text, ""])

OUT_TXT.write_text("\n".join(lines), encoding="utf-8")
OUT_JSON.write_text(json_text + "\n", encoding="utf-8")
print("Wrote:", OUT_TXT)
print("Wrote:", OUT_JSON)


## 9) Quick verification
Read first lines of outputs to verify they were generated in this run.


In [ ]:
print("TXT preview:\n")
print("\n".join(OUT_TXT.read_text(encoding="utf-8").splitlines()[:35]))

print("\nJSON preview:\n")
print("\n".join(OUT_JSON.read_text(encoding="utf-8").splitlines()[:35]))
